In [1]:
import csv

# 인구 데이터 읽어와서 리스트로 저장하기
f1 = open('LOCAL_PEOPLE_DONG_201912.csv', encoding='utf8')    
data = csv.reader(f1)
next(data)
data = list(data)

# 행정동코드 데이터 불러와서 리스트로 저장하기
f2 = open('dong_code.csv', encoding='utf8')
code_data = csv.reader(f2)
next(code_data)
next(code_data)
code_data = list(code_data)

# 위치 데이터 불러와서 리스트로 저장하기
f3 = open('dong_location.csv')
location_data = csv.reader(f3)
next(location_data)
location_data = list(location_data)

# 인구데이터의 데이터타입 변경하기 
for row in data : 
    for i in range(1, 32) : 
        if i <= 2 : 
            row[i] = int(row[i])
        else : 
            row[i] = float(row[i])
            
# 행정동코드 데이터의 데이터 타입 변환하기
for row in code_data :
    row[0], row[1] = int(row[0]), int(row[1]) # 7자리 행정동코드, 8자리 행정동코드

# 위치 정보 데이터의 데이터 타입 변환하기
for row in location_data :
    row[2], row[-2], row[-1] = int(row[2]), float(row[-2]), float(row[-1])


# 2. 행정동별 인구수를 저장하는 빈 딕셔너리를 만들고 행정동별 총생활인구를 딕셔너리에 저장하기
dong_population = {}

for row in data: 
    dong_code, p = row[2], row[3]
    if dong_code not in dong_population.keys(): 
        dong_population[dong_code] = p
    else:
        dong_population[dong_code] += p
        
# print(dong_population) 


# 3. 인구수를 기준으로 내림차순 정렬하고, 앞에서 10개 항목만 별도로 저장하기
import operator

top10 = sorted(dong_population.items(), key=operator.itemgetter(1), reverse = True)[:10]
#print(top10)

top10_code8 = [t[0] for t in top10]
print(top10_code8)


top10_name = [0 for i in range(10)]
top10_lat  = [0 for i in range(10)]
top10_long = [0 for i in range(10)]

# 4. 8자리 행정동코드에 해당하는 행정동명 찾기
for i in range(len(top10_code8)) : 
    for row in code_data : 
        if row[1] == top10_code8[i] :  
            # 행정동명 리스트에 저장하기 
            top10_name[i] = row[-1]
    # 5. 행정동명에 해당하는 위도와 경도를 찾아 리스트에 저장하기
    for row in location_data : 
        if row[3] == top10_name[i] :
            top10_lat[i], top10_long[i] = row[-1], row[-2] 

print(top10_lat)
print(top10_long)


# 추가. 행정동명으로 위치 정보를 찾지 못하면 7자리 행정동코드로 찾기
for i in range(len(top10_lat)):
    # 위도나 경도가 0인 요소가 있다면
    if top10_lat[i] == 0 or top10_long[i] == 0 :
        # code_data에서 7자리 행정동코드를 구해 dong7에 저장하기
        for row in code_data :
            if top10_code8[i] == row[1] :
                dong7 = row[0]
        # location_data에서 7자리 행정동코드가 dong7인 행을 찾아 위도와 경도를 구하기
        for row in location_data : 
            if dong7 == row[2] :
                top10_lat[i], top10_long[i] = row[-1], row[-2] 

print(top10_lat)
print(top10_long)


# 6. 완성된 리스트 지도에 표시하기
import folium

for i in range(len(top10_name)) : 
    top10_name[i] = str(i+1)+'. '+top10_name[i]

    
map_top10 = folium.Map([top10_lat[0], top10_long[0]], zoom_start=12)
for i in range(10): 
    folium.Marker(
        [top10_lat[i], top10_long[i]], tooltip=top10_name[i],
        icon=folium.Icon(color='red', icon='star')
        ).add_to(map_top10)

map_top10
#-------------------------------------------------------------------------------------------------
# 추가 분석 : 남성이 많은 지역과 여성이 많은 지역 비교하기
import csv

# 인구 데이터 읽어와서 리스트로 저장하기
f1 = open('LOCAL_PEOPLE_DONG_201912.csv', encoding='utf8')    
data = csv.reader(f1)
next(data)
data = list(data)

# 행정동코드 데이터 불러와서 리스트로 저장하기
f2 = open('dong_code.csv', encoding='utf8')
code_data = csv.reader(f2)
next(code_data)
next(code_data)
code_data = list(code_data)

# 위치 데이터 불러와서 리스트로 저장하기
f3 = open('dong_location.csv')
location_data = csv.reader(f3)
next(location_data)
location_data = list(location_data)

# 인구데이터의 데이터타입 변경하기 
for row in data : 
    for i in range(1, 32) : 
        if i <= 2 : 
            row[i] = int(row[i])
        else : 
            row[i] = float(row[i])
            
# 행정동코드 데이터의 데이터 타입 변환하기
for row in code_data :
    row[0], row[1] = int(row[0]), int(row[1]) # 7자리 행정동코드, 8자리 행정동코드

# 위치 정보 데이터의 데이터 타입 변환하기
for row in location_data :
    row[2], row[-2], row[-1] = int(row[2]), float(row[-2]), float(row[-1])

# 2. 인구 데이터를 읽으며 행정동별 남성 인구수를 저장하는 딕셔너리 완성하기
dong_population = {}

for row in data: 
    # 남성 인구수를 저장하는 변수 p
    dong_code, p = row[2], sum(row[4:18])
    if dong_code not in dong_population.keys(): 
        dong_population[dong_code] = p
    else:
        dong_population[dong_code] += p
        
# print(dong_population) 

# 3. 인구수를 기준으로 내림차순 정렬하고, 앞에서 10개 항목만 별도로 저장하기
import operator

top10 = sorted(dong_population.items(), key=operator.itemgetter(1), reverse = True)[:10]
#print(top10)

top10_code8 = [t[0] for t in top10]
print(top10_code8)


top10_name = [0 for i in range(10)]
top10_lat  = [0 for i in range(10)]
top10_long = [0 for i in range(10)]

# 4. 8자리 행정동코드에 해당하는 행정동명 찾기
for i in range(len(top10_code8)) : 
    for row in code_data : 
        if row[1] == top10_code8[i] :  
            # 행정동명 리스트에 저장하기 
            top10_name[i] = row[-1]
    # 5. 행정동명에 해당하는 위도와 경도를 찾아 리스트에 저장하기
    for row in location_data : 
        if row[3] == top10_name[i] :
            top10_lat[i], top10_long[i] = row[-1], row[-2] 

print(top10_lat)
print(top10_long)


# 추가. 행정동명으로 위치 정보를 찾지 못하면 7자리 행정동코드로 찾기
for i in range(len(top10_lat)):
    # 위도나 경도가 0인 요소가 있다면
    if top10_lat[i] == 0 or top10_long[i] == 0 :
        # code_data에서 7자리 행정동코드를 구해 dong7에 저장하기
        for row in code_data :
            if top10_code8[i] == row[1] :
                dong7 = row[0]
        # location_data에서 7자리 행정동코드가 dong7인 행을 찾아 위도와 경도를 구하기
        for row in location_data : 
            if dong7 == row[2] :
                top10_lat[i], top10_long[i] = row[-1], row[-2] 

print(top10_lat)
print(top10_long)


# 6. 완성된 리스트 지도에 표시하기
import folium

for i in range(len(top10_name)) : 
    top10_name[i] = str(i+1)+'. '+top10_name[i]

    
map_top10 = folium.Map([top10_lat[0], top10_long[0]], zoom_start=12) 
for i in range(10):
    # 1~5위 지역은 진한 파란색, 6~7위 지역은 연한 파란색으로 표시하기
    if i < 5: 
        folium.Marker(
            [top10_lat[i], top10_long[i]], tooltip=top10_name[i], 
            icon=folium.Icon(color='darkblue', icon='male', prefix='fa')
        ).add_to(map_top10)
    else: 
        folium.Marker(
            [top10_lat[i], top10_long[i]], tooltip=top10_name[i], 
            icon=folium.Icon(color='lightblue', icon='male', prefix='fa') 
        ).add_to(map_top10)

map_top10


#-------------------------------------------------------------------------
# 2. 인구 데이터를 읽으며 행정동별 여성 인구수를 저장하는 딕셔너리 완성하기
dong_population = {}

for row in data: 
    # 여성 인구수를 저장하는 변수 p
    dong_code, p = row[2], sum(row[18:32])
    if dong_code not in dong_population.keys(): 
        dong_population[dong_code] = p
    else:
        dong_population[dong_code] += p
        
# print(dong_population) 


# 3. 인구수를 기준으로 내림차순 정렬하고, 앞에서 10개 항목만 별도로 저장하기
import operator

top10 = sorted(dong_population.items(), key=operator.itemgetter(1), reverse = True)[:10]
#print(top10)

top10_code8 = [t[0] for t in top10]
print(top10_code8)


top10_name = [0 for i in range(10)]
top10_lat  = [0 for i in range(10)]
top10_long = [0 for i in range(10)]

# 4. 8자리 행정동코드에 해당하는 행정동명 찾기
for i in range(len(top10_code8)) : 
    for row in code_data : 
        if row[1] == top10_code8[i] :  
            # 행정동명 리스트에 저장하기 
            top10_name[i] = row[-1]
    # 5. 행정동명에 해당하는 위도와 경도를 찾아 리스트에 저장하기
    for row in location_data : 
        if row[3] == top10_name[i] :
            top10_lat[i], top10_long[i] = row[-1], row[-2] 

print(top10_lat)
print(top10_long)

# 추가. 행정동명으로 위치 정보를 찾지 못하면 7자리 행정동코드로 찾기
for i in range(len(top10_lat)):
    # 위도나 경도가 0인 요소가 있다면
    if top10_lat[i] == 0 or top10_long[i] == 0 :
        # code_data에서 7자리 행정동코드를 구해 dong7에 저장하기
        for row in code_data :
            if top10_code8[i] == row[1] :
                dong7 = row[0]
        # location_data에서 7자리 행정동코드가 dong7인 행을 찾아 위도와 경도를 구하기
        for row in location_data : 
            if dong7 == row[2] :
                top10_lat[i], top10_long[i] = row[-1], row[-2] 

print(top10_lat)
print(top10_long)

# 6. 완성된 리스트 지도에 표시하기
import folium

for i in range(len(top10_name)) : 
    top10_name[i] = str(i+1)+'. '+top10_name[i]

    
map_top10 = folium.Map([top10_lat[0], top10_long[0]], zoom_start=12) 
for i in range(10):
    # 1~5위 지역은 진한 파란색, 6~7위 지역은 연한 파란색으로 표시하기
    if i < 5: 
        folium.Marker(
            [top10_lat[i], top10_long[i]], tooltip=top10_name[i], 
            icon=folium.Icon(color='red', icon='female', prefix='fa')
        ).add_to(map_top10)
    else: 
        folium.Marker(
            [top10_lat[i], top10_long[i]], tooltip=top10_name[i], 
            icon=folium.Icon(color='pink', icon='female', prefix='fa') 
        ).add_to(map_top10)

map_top10

[11680640, 11560540, 11440660, 11110615, 11650530, 11410585, 11380690, 11545510, 11680650, 11680610]
[37.4999211, 0, 37.5550337, 37.5777701, 37.4856591, 37.5659638, 37.64193, 37.4756389, 37.4984733, 37.4955294]
[127.0367584, 0, 126.9228533, 126.9892737, 127.0097616, 126.9384315, 126.9387277, 126.8850721, 127.0449272, 127.0662436]
[37.4999211, 37.5286662, 37.5550337, 37.5777701, 37.4856591, 37.5659638, 37.64193, 37.4756389, 37.4984733, 37.4955294]
[127.0367584, 126.9290731, 126.9228533, 126.9892737, 127.0097616, 126.9384315, 126.9387277, 126.8850721, 127.0449272, 127.0662436]
[11680640, 11560540, 11110615, 11440660, 11650530, 11545510, 11380690, 11680610, 11410585, 11680650]
[37.4999211, 0, 37.5777701, 37.5550337, 37.4856591, 37.4756389, 37.64193, 37.4955294, 37.5659638, 37.4984733]
[127.0367584, 0, 126.9892737, 126.9228533, 127.0097616, 126.8850721, 126.9387277, 127.0662436, 126.9384315, 127.0449272]
[37.4999211, 37.5286662, 37.5777701, 37.5550337, 37.4856591, 37.4756389, 37.64193, 37.